# GPT Model Finetuning & Inference

This notebook handles the complete workflow for finetuning and running inference with OpenAI GPT models.

## Workflow
1. **Prepare training data** in OpenAI's JSONL format
2. **Upload data** to OpenAI
3. **Create finetuning job** and wait for completion
4. **Run inference** with finetuned models

## Models
- GPT-3.5-turbo
- GPT-4o, GPT-4o-mini
- GPT-4.1, GPT-4.1-mini

## Tasks
- Bandgap (regression)
- Dielectric (regression)
- CrystalStructure (classification)
- MatKG (link prediction)

**Note**: GPT models use a different prompt format (system/user messages) compared to open-source models.

## 1. Setup & Configuration

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import pickle
import time
from tqdm import tqdm
from openai import OpenAI
from collections import Counter
from scipy.stats import entropy

# For MatKG evaluation
try:
    from sentence_transformers import SentenceTransformer, util
    emb_model = SentenceTransformer('all-MiniLM-L6-v2')
except ImportError:
    print("sentence-transformers not installed. Install with: pip install sentence-transformers")
    emb_model = None

# API Key - set your own key here or use environment variable
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your_openai_key_here')
client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client initialized")

In [ ]:
# Directory Configuration
DATA_DIR = '../data'
RESULTS_DIR = '../results'
FT_DATA_DIR = '../data/finetuning_jsonl'  # For prepared JSONL files

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FT_DATA_DIR, exist_ok=True)

# Data paths
TRAIN_DATA = {
    'Bandgap': f'{DATA_DIR}/Bandgap/Bandgap_train_set.csv',
    'Dielectric': f'{DATA_DIR}/Dielectric/Dielectric_train_set.csv',
    'CrystalStructure': f'{DATA_DIR}/CrystalStructure/Crystal_train_set.csv',
    'MatKG': f'{DATA_DIR}/MatKG/MatKG_train_df.csv'
}

TEST_DATA = {
    'Bandgap': f'{DATA_DIR}/Bandgap/Bandgap_test_set.csv',
    'Dielectric': f'{DATA_DIR}/Dielectric/Dielectric_test_set.csv',
    'CrystalStructure': f'{DATA_DIR}/CrystalStructure/Crystal_test_set.csv',
    'MatKG': f'{DATA_DIR}/MatKG/MatKG_test_df.csv'
}

# Base models available for finetuning
BASE_MODELS = {
    'gpt-3.5-turbo': 'gpt-3.5-turbo-1106',
    'gpt-4o': 'gpt-4o-2024-08-06',
    'gpt-4o-mini': 'gpt-4o-mini-2024-07-18',
    'gpt-4.1': 'gpt-4.1-2025-04-14',
    'gpt-4.1-mini': 'gpt-4.1-mini-2025-04-14'
}

# Number of inference iterations
N_ITERATIONS = 10

## 2. System Prompts for Each Task

GPT models use system prompts to define the task context.

In [ ]:
SYSTEM_PROMPTS = {
    'Bandgap': """You are an expert in the electronic bandstructure of materials. 
For any given chemical formula of a material, you can accurately provide the numerical value 
of its bandgap in electron volts (eV). The output should only be a list of floating point 
numbers separated by commas enclosed in square brackets. Do not include any other alphabets 
or characters. There will only be as many floating point numbers as materials.""",

    'Dielectric': """You are an expert in the electronic properties of materials. 
For any given chemical formula of a material, you can accurately provide the numerical value 
of its dielectric constant. The output should only be a list of floating point numbers 
separated by commas enclosed in square brackets. Do not include any other alphabets or 
characters. There will only be as many floating point numbers as materials.""",

    'CrystalStructure': """You are an expert in the crystal structure of materials. 
For any given chemical formula of a material, you can accurately provide the Crystal Structure. 
The output should be a list of strings selected from one of these: 
['Trigonal', 'Cubic', 'Tetragonal', 'Hexagonal', 'Monoclinic', 'Orthorhombic', 'Triclinic']. 
There will only be as many responses as there are materials.""",

    'MatKG': """You are a helpful scientific assistant trained in Materials Science."""
}

## 3. Prepare Training Data in JSONL Format

OpenAI finetuning requires data in JSONL format with specific structure.

In [ ]:
def prepare_finetuning_data(task, output_path=None):
    """
    Convert training data to OpenAI JSONL format.
    
    OpenAI format:
    {"messages": [{"role": "system", "content": "..."}, 
                  {"role": "user", "content": "..."}, 
                  {"role": "assistant", "content": "..."}]}
    
    Args:
        task: Task name
        output_path: Path to save JSONL file
    
    Returns:
        Path to created JSONL file
    """
    if output_path is None:
        output_path = f'{FT_DATA_DIR}/{task}_train.jsonl'
    
    train_df = pd.read_csv(TRAIN_DATA[task])
    system_prompt = SYSTEM_PROMPTS[task]
    
    with open(output_path, 'w') as f:
        for idx, row in train_df.iterrows():
            # Get prompt and response columns
            # Adjust column names based on your data format
            if 'Prompt' in row:
                user_content = row['Prompt']
            elif 'prompt' in row:
                user_content = row['prompt']
            else:
                user_content = row.iloc[0]  # First column
            
            if 'Response' in row:
                assistant_content = str(row['Response'])
            elif 'response' in row:
                assistant_content = str(row['response'])
            elif 'completion' in row:
                assistant_content = str(row['completion'])
            else:
                assistant_content = str(row.iloc[1])  # Second column
            
            # Create message structure
            entry = {
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": assistant_content}
                ]
            }
            
            f.write(json.dumps(entry) + '\n')
    
    print(f"Created {output_path} with {len(train_df)} examples")
    return output_path

In [ ]:
def validate_jsonl(filepath):
    """
    Validate JSONL file format for OpenAI finetuning.
    """
    errors = []
    with open(filepath, 'r') as f:
        for i, line in enumerate(f):
            try:
                data = json.loads(line)
                if 'messages' not in data:
                    errors.append(f"Line {i+1}: Missing 'messages' key")
                else:
                    roles = [m.get('role') for m in data['messages']]
                    if 'system' not in roles:
                        errors.append(f"Line {i+1}: Missing system message")
                    if 'user' not in roles:
                        errors.append(f"Line {i+1}: Missing user message")
                    if 'assistant' not in roles:
                        errors.append(f"Line {i+1}: Missing assistant message")
            except json.JSONDecodeError as e:
                errors.append(f"Line {i+1}: Invalid JSON - {e}")
    
    if errors:
        print(f"Validation errors in {filepath}:")
        for e in errors[:10]:  # Show first 10 errors
            print(f"  {e}")
        return False
    else:
        print(f"✓ {filepath} is valid")
        return True

In [ ]:
# Prepare training data for all tasks
# Uncomment to run

# for task in ['Bandgap', 'Dielectric', 'CrystalStructure', 'MatKG']:
#     jsonl_path = prepare_finetuning_data(task)
#     validate_jsonl(jsonl_path)

## 4. Upload Training Data to OpenAI

In [ ]:
def upload_training_file(filepath):
    """
    Upload training file to OpenAI.
    
    Returns:
        File ID for use in finetuning
    """
    with open(filepath, 'rb') as f:
        response = client.files.create(
            file=f,
            purpose='fine-tune'
        )
    
    file_id = response.id
    print(f"Uploaded {filepath}")
    print(f"File ID: {file_id}")
    return file_id


def list_uploaded_files():
    """
    List all uploaded files in your OpenAI account.
    """
    files = client.files.list()
    print("Uploaded files:")
    for f in files.data:
        print(f"  {f.id}: {f.filename} ({f.purpose})")
    return files

In [ ]:
# Upload training files
# Uncomment to run

# uploaded_files = {}
# for task in ['Bandgap', 'Dielectric', 'CrystalStructure', 'MatKG']:
#     jsonl_path = f'{FT_DATA_DIR}/{task}_train.jsonl'
#     if os.path.exists(jsonl_path):
#         file_id = upload_training_file(jsonl_path)
#         uploaded_files[task] = file_id
# 
# print("\nUploaded file IDs:")
# for task, fid in uploaded_files.items():
#     print(f"  {task}: {fid}")

## 5. Create Finetuning Jobs

In [ ]:
def create_finetuning_job(training_file_id, base_model, suffix=None, n_epochs=3):
    """
    Create a finetuning job.
    
    Args:
        training_file_id: OpenAI file ID of training data
        base_model: Base model to finetune (e.g., 'gpt-4o-2024-08-06')
        suffix: Custom suffix for the finetuned model name
        n_epochs: Number of training epochs
    
    Returns:
        Job ID
    """
    job = client.fine_tuning.jobs.create(
        training_file=training_file_id,
        model=base_model,
        suffix=suffix,
        hyperparameters={
            "n_epochs": n_epochs
        }
    )
    
    print(f"Created finetuning job: {job.id}")
    print(f"Status: {job.status}")
    return job.id


def check_job_status(job_id):
    """
    Check the status of a finetuning job.
    """
    job = client.fine_tuning.jobs.retrieve(job_id)
    print(f"Job {job_id}:")
    print(f"  Status: {job.status}")
    print(f"  Model: {job.fine_tuned_model}")
    if job.error:
        print(f"  Error: {job.error}")
    return job


def wait_for_job(job_id, poll_interval=60):
    """
    Wait for a finetuning job to complete.
    
    Args:
        job_id: Job ID to monitor
        poll_interval: Seconds between status checks
    
    Returns:
        Finetuned model ID if successful, None otherwise
    """
    print(f"Waiting for job {job_id} to complete...")
    
    while True:
        job = client.fine_tuning.jobs.retrieve(job_id)
        status = job.status
        
        print(f"  Status: {status}")
        
        if status == 'succeeded':
            print(f"\n✓ Job completed successfully!")
            print(f"  Finetuned model: {job.fine_tuned_model}")
            return job.fine_tuned_model
        elif status in ['failed', 'cancelled']:
            print(f"\n✗ Job {status}")
            if job.error:
                print(f"  Error: {job.error}")
            return None
        
        time.sleep(poll_interval)


def list_finetuning_jobs():
    """
    List all finetuning jobs.
    """
    jobs = client.fine_tuning.jobs.list(limit=20)
    print("Recent finetuning jobs:")
    for job in jobs.data:
        print(f"  {job.id}: {job.status} -> {job.fine_tuned_model}")
    return jobs

In [ ]:
# Example: Create finetuning job for Bandgap with GPT-4o
# Uncomment to run

# job_id = create_finetuning_job(
#     training_file_id='file-xxxxx',  # Replace with your file ID
#     base_model='gpt-4o-2024-08-06',
#     suffix='bandgap',
#     n_epochs=3
# )

# # Wait for completion
# finetuned_model = wait_for_job(job_id)

In [ ]:
def finetune_all_models(uploaded_files, base_models=None, n_epochs=3):
    """
    Finetune all base models on all tasks.
    
    Args:
        uploaded_files: Dict of {task: file_id}
        base_models: List of base model names to finetune (default: all)
        n_epochs: Number of training epochs
    
    Returns:
        Dict of {task: {model_name: finetuned_model_id}}
    """
    if base_models is None:
        base_models = list(BASE_MODELS.keys())
    
    finetuned_models = {}
    
    for task, file_id in uploaded_files.items():
        finetuned_models[task] = {}
        
        for model_name in base_models:
            base_model = BASE_MODELS[model_name]
            suffix = f"{task.lower()}-{model_name.replace('.', '-')}"
            
            print(f"\n{'='*60}")
            print(f"Finetuning {model_name} on {task}")
            
            try:
                job_id = create_finetuning_job(
                    training_file_id=file_id,
                    base_model=base_model,
                    suffix=suffix,
                    n_epochs=n_epochs
                )
                
                finetuned_model = wait_for_job(job_id)
                finetuned_models[task][model_name] = finetuned_model
                
            except Exception as e:
                print(f"Error finetuning {model_name} on {task}: {e}")
                finetuned_models[task][model_name] = None
    
    return finetuned_models

# Example usage:
# finetuned_models = finetune_all_models(uploaded_files, base_models=['gpt-4o', 'gpt-4o-mini'])

## 6. Store Finetuned Model IDs

After finetuning, save your model IDs for later use.

In [ ]:
# Template for storing finetuned model IDs
# Fill in your own model IDs after finetuning

FINETUNED_MODELS = {
    'Bandgap': {
        'gpt-3.5-turbo': None,  # e.g., 'ft:gpt-3.5-turbo-1106:org:suffix:id'
        'gpt-4o': None,
        'gpt-4o-mini': None,
        'gpt-4.1': None,
        'gpt-4.1-mini': None
    },
    'Dielectric': {
        'gpt-3.5-turbo': None,
        'gpt-4o': None,
        'gpt-4o-mini': None,
        'gpt-4.1': None,
        'gpt-4.1-mini': None
    },
    'CrystalStructure': {
        'gpt-3.5-turbo': None,
        'gpt-4o': None,
        'gpt-4o-mini': None,
        'gpt-4.1': None,
        'gpt-4.1-mini': None
    },
    'MatKG': {
        'gpt-3.5-turbo': None,
        'gpt-4o': None,
        'gpt-4o-mini': None,
        'gpt-4.1': None,
        'gpt-4.1-mini': None
    }
}

def save_model_ids(finetuned_models, filepath='finetuned_model_ids.json'):
    """Save finetuned model IDs to JSON file."""
    with open(filepath, 'w') as f:
        json.dump(finetuned_models, f, indent=2)
    print(f"Saved model IDs to {filepath}")

def load_model_ids(filepath='finetuned_model_ids.json'):
    """Load finetuned model IDs from JSON file."""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            return json.load(f)
    return FINETUNED_MODELS

## 7. Task-Specific Batcher Classes for Inference

Each task has a specific prompt format and parsing logic.

In [ ]:
class BandGap_Batcher:
    """Batch predictor for Bandgap regression task."""
    
    def __init__(self, index, df, model):
        self.index = index
        self.df = df
        self.model = model
        
        # Parse ground truth
        self.ground = self._parse_list(self.df.loc[index, 'Response'])
        self.ground = [float(x) for x in self.ground]
        
        # Parse metadata
        self.mpids = ['mp-' + x.strip().replace("'", "") 
                      for x in self._parse_list(self.df.loc[index, 'mpid'])]
        self.formulas = [x.strip().replace("'", "") 
                         for x in self._parse_list(self.df.loc[index, 'Formula'])]
        
        self.prompt = self.df.loc[index, 'Prompt']
        self.preds = self.predict()
    
    def _parse_list(self, val):
        """Parse string list representation."""
        if isinstance(val, str):
            return val.replace('[', '').replace(']', '').split(',')
        return val
    
    def predict(self):
        """Run GPT prediction."""
        completion = client.chat.completions.create(
            model=self.model,
            temperature=0.1,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPTS['Bandgap']},
                {"role": "user", "content": self.prompt}
            ]
        )
        results = completion.choices[0].message.content
        results = results.replace('[', '').replace(']', '')
        try:
            return [float(x.strip()) for x in results.split(',')]
        except:
            return [None] * len(self.ground)

In [ ]:
class Dielectric_Batcher:
    """Batch predictor for Dielectric regression task."""
    
    def __init__(self, index, df, model):
        self.index = index
        self.df = df
        self.model = model
        
        self.ground = self._parse_list(self.df.loc[index, 'Response'])
        self.ground = [float(x) for x in self.ground]
        
        self.mpids = ['mp-' + x.strip().replace("'", "") 
                      for x in self._parse_list(self.df.loc[index, 'mpid'])]
        self.formulas = [x.strip().replace("'", "") 
                         for x in self._parse_list(self.df.loc[index, 'Formula'])]
        
        self.prompt = self.df.loc[index, 'Prompt']
        self.preds = self.predict()
    
    def _parse_list(self, val):
        if isinstance(val, str):
            return val.replace('[', '').replace(']', '').split(',')
        return val
    
    def predict(self):
        completion = client.chat.completions.create(
            model=self.model,
            temperature=0.1,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPTS['Dielectric']},
                {"role": "user", "content": self.prompt}
            ]
        )
        results = completion.choices[0].message.content
        results = results.replace('[', '').replace(']', '')
        try:
            return [float(x.strip()) for x in results.split(',')]
        except:
            return [None] * len(self.ground)

In [ ]:
class Crystal_Batcher:
    """Batch predictor for Crystal Structure classification task."""
    
    def __init__(self, index, df, model):
        self.index = index
        self.df = df
        self.model = model
        
        self.ground = self._parse_list(self.df.loc[index, 'Response'])
        self.ground = [x.strip().strip("'") for x in self.ground]
        
        self.mpids = ['mp-' + x.strip().replace("'", "") 
                      for x in self._parse_list(self.df.loc[index, 'mpid'])]
        self.formulas = [x.strip().replace("'", "") 
                         for x in self._parse_list(self.df.loc[index, 'Formula'])]
        
        self.prompt = self.df.loc[index, 'Prompt']
        self.preds = self.predict()
    
    def _parse_list(self, val):
        if isinstance(val, str):
            return val.replace('[', '').replace(']', '').split(',')
        return val
    
    def predict(self):
        user_prompt = f"""[INSTRUCTION] Please provide the crystal structure for the following materials.
Output the values as a comma-separated list. Do not include the names of the materials in your response.
Do not include period at the end of response. Choose from one of the responses below:
['Trigonal', 'Cubic', 'Tetragonal', 'Hexagonal', 'Monoclinic', 'Orthorhombic', 'Triclinic']
Example:
Materials: ['Ca(GaAs)2', 'KGe', 'KSn2Cl5', 'K3NdSi2O7', 'LiCaAlN2'],
Correct Response: ['Cubic', 'Tetragonal', 'Hexagonal', 'Triclinic', 'Cubic']
Now, provide the crystal structure for these materials: {self.formulas}[/INSTRUCTION]"""
        
        completion = client.chat.completions.create(
            model=self.model,
            temperature=0.1,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPTS['CrystalStructure']},
                {"role": "user", "content": user_prompt}
            ]
        )
        results = completion.choices[0].message.content
        results = results.replace('[', '').replace(']', '')
        return [x.strip().strip("'") for x in results.split(',')]

In [ ]:
class EntityEvaluator:
    """Evaluator for MatKG link prediction task."""
    
    def __init__(self, subject, rel, prompt, model, kg_df):
        self.ent = subject
        self.sub = rel.split('-')[0]
        self.obj = rel.split('-')[1]
        self.kg_df = kg_df
        self.df = self._get_entity_df()
        self.prompt = prompt
        self.model = model
        
        if emb_model is not None:
            self.obj_emb = emb_model.encode(self.df['Object'].tolist(), convert_to_tensor=True)
        else:
            self.obj_emb = None
        
        self.api_results = self._get_api_results()
        self.evals = self._evaluate()
    
    def _get_entity_df(self):
        return self.kg_df[
            (self.kg_df['Subject'] == self.ent) & 
            (self.kg_df['Rel'] == f"{self.sub}-{self.obj}")
        ].sort_values('Count', ascending=False)
    
    def _get_api_results(self):
        completion = client.chat.completions.create(
            model=self.model,
            temperature=0.1,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPTS['MatKG']},
                {"role": "user", "content": self.prompt}
            ]
        )
        results = completion.choices[0].message.content
        result_list = results.replace('[', '').replace(']', '').split(',')
        return [r.strip().strip("'") for r in result_list]
    
    def _evaluate(self, threshold=0.8):
        if self.obj_emb is None:
            return {}
        
        mkg_count_dict = {}
        
        for query in self.api_results:
            top_5, top_1 = 0, 0
            query_emb = emb_model.encode(query, convert_to_tensor=True)
            cos_sim = util.pytorch_cos_sim(query_emb, self.obj_emb)[0].cpu()
            
            similar_mask = (cos_sim > threshold).tolist()
            similar_objects = self.df[similar_mask]
            
            if any(cos_sim[:5] > threshold):
                top_5 = 1
            if len(cos_sim) > 0 and cos_sim[0] > threshold:
                top_1 = 1
            
            mkg_count_dict[query] = {
                'count': similar_objects['Count'].sum() if len(similar_objects) > 0 else 0,
                'top_5': top_5,
                'top_1': top_1
            }
        
        return mkg_count_dict

## 8. Run Inference Experiments

In [ ]:
def run_gpt_experiment(task, model_name, model_id, n_iterations=10, delay=1.0):
    """
    Run GPT inference experiment.
    
    Args:
        task: Task name ('Bandgap', 'Dielectric', 'CrystalStructure', 'MatKG')
        model_name: Model short name for output file naming
        model_id: Full model ID (base or finetuned)
        n_iterations: Number of inference iterations
        delay: Delay between API calls to avoid rate limits
    """
    # Load test data
    test_df = pd.read_csv(TEST_DATA[task])
    
    # Select batcher class
    if task == 'Bandgap':
        BatcherClass = BandGap_Batcher
    elif task == 'Dielectric':
        BatcherClass = Dielectric_Batcher
    elif task == 'CrystalStructure':
        BatcherClass = Crystal_Batcher
    else:  # MatKG
        kg_df = pd.read_csv(f'{DATA_DIR}/MatKG/SUBRELOBJ.csv')
    
    # Determine if base or finetuned
    is_finetuned = model_id.startswith('ft:')
    ft_label = task if is_finetuned else 'base'
    
    for iteration in range(n_iterations):
        # Output file path
        if task == 'MatKG':
            output_file = f'{RESULTS_DIR}/{task}_{model_name}_ft_{ft_label}_iter_{iteration}.pickle'
        else:
            output_file = f'{RESULTS_DIR}/{task}_{model_name}_ft_{ft_label}_iter_{iteration}.csv'
        
        if os.path.exists(output_file):
            print(f"Skipping (exists): {output_file}")
            continue
        
        print(f"Task: {task} | Model: {model_name} | FT: {ft_label} | Iter: {iteration}")
        
        results = {}
        
        for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
            try:
                if task in ['Bandgap', 'Dielectric', 'CrystalStructure']:
                    batcher = BatcherClass(index, test_df, model_id)
                    for mpid, formula, ground, pred in zip(
                        batcher.mpids, batcher.formulas, batcher.ground, batcher.preds
                    ):
                        results[mpid] = {
                            'Formula': formula,
                            'Ground': ground,
                            'Prediction': pred
                        }
                else:  # MatKG
                    ent = row['Subject']
                    rel = row['Rel']
                    prompt = row['Prompt']
                    
                    if ent not in results:
                        results[ent] = {}
                    
                    evaluator = EntityEvaluator(ent, rel, prompt, model_id, kg_df)
                    results[ent][rel] = evaluator.evals
                
                time.sleep(delay)  # Rate limiting
                
            except Exception as e:
                print(f"Error at index {index}: {e}")
                time.sleep(5)  # Longer delay on error
        
        # Save results
        if task == 'MatKG':
            with open(output_file, 'wb') as f:
                pickle.dump(results, f)
        else:
            pd.DataFrame.from_dict(results, orient='index').to_csv(output_file)
        
        print(f"Saved: {output_file}")

In [ ]:
# Example: Run experiments
# Uncomment to run

# # Base model inference
# run_gpt_experiment(
#     task='Bandgap',
#     model_name='gpt-4o',
#     model_id=BASE_MODELS['gpt-4o'],
#     n_iterations=10
# )

# # Finetuned model inference (replace with your finetuned model ID)
# run_gpt_experiment(
#     task='Bandgap',
#     model_name='gpt-4o',
#     model_id='ft:gpt-4o-2024-08-06:org:bandgap:xxxx',  # Your finetuned model ID
#     n_iterations=10
# )

In [ ]:
def run_all_experiments(finetuned_models=None):
    """
    Run all base and finetuned experiments.
    
    Args:
        finetuned_models: Dict of finetuned model IDs
    """
    if finetuned_models is None:
        finetuned_models = load_model_ids()
    
    for model_name, base_model_id in BASE_MODELS.items():
        for task in ['Bandgap', 'Dielectric', 'CrystalStructure', 'MatKG']:
            print(f"\n{'='*60}")
            
            # Base inference
            print(f"Running BASE: {model_name} | {task}")
            run_gpt_experiment(
                task=task,
                model_name=model_name,
                model_id=base_model_id,
                n_iterations=N_ITERATIONS
            )
            
            # Finetuned inference (if available)
            ft_model_id = finetuned_models.get(task, {}).get(model_name)
            if ft_model_id:
                print(f"Running FINETUNED: {model_name} | {task}")
                run_gpt_experiment(
                    task=task,
                    model_name=model_name,
                    model_id=ft_model_id,
                    n_iterations=N_ITERATIONS
                )
            else:
                print(f"Skipping finetuned (no model ID): {model_name} | {task}")

# Uncomment to run all
# run_all_experiments()

## 9. Aggregate Results into Summary Format

In [ ]:
def aggregate_gpt_results(task, model_name, ft_label, n_iterations=10):
    """
    Aggregate results from multiple iterations into summary format.
    """
    all_results = {}
    
    for iteration in range(n_iterations):
        if task == 'MatKG':
            filepath = f'{RESULTS_DIR}/{task}_{model_name}_ft_{ft_label}_iter_{iteration}.pickle'
            if not os.path.exists(filepath):
                continue
            with open(filepath, 'rb') as f:
                iter_results = pickle.load(f)
        else:
            filepath = f'{RESULTS_DIR}/{task}_{model_name}_ft_{ft_label}_iter_{iteration}.csv'
            if not os.path.exists(filepath):
                continue
            iter_results = pd.read_csv(filepath, index_col=0).to_dict('index')
        
        # Merge into all_results
        for key, val in iter_results.items():
            if key not in all_results:
                all_results[key] = {'predictions': []}
                if task != 'MatKG':
                    all_results[key]['Formula'] = val.get('Formula')
                    all_results[key]['Ground'] = val.get('Ground')
            
            if task != 'MatKG':
                all_results[key]['predictions'].append(val.get('Prediction'))
    
    if len(all_results) == 0:
        print(f"No results found for {task}_{model_name}_ft_{ft_label}")
        return None
    
    # Create summary DataFrame
    if task != 'MatKG':
        summary_records = []
        for mpid, data in all_results.items():
            record = {
                'mpid': mpid,
                'Formula': data['Formula'],
                'Ground': data['Ground']
            }
            
            preds = [p for p in data['predictions'] if p is not None]
            
            # Add individual predictions
            for i, p in enumerate(data['predictions']):
                record[f'prediction_{i+1}'] = p
            
            # Mode prediction and entropy
            if task in ['Bandgap', 'Dielectric']:
                if preds:
                    record['mode_prediction'] = np.median(preds)
                    bins = np.histogram_bin_edges(preds, bins='auto')
                    hist, _ = np.histogram(preds, bins=bins)
                    record['prediction_entropy'] = entropy(hist + 1e-10)
                else:
                    record['mode_prediction'] = None
                    record['prediction_entropy'] = None
            else:  # Classification
                if preds:
                    counter = Counter(preds)
                    record['mode_prediction'] = counter.most_common(1)[0][0]
                    record['prediction_entropy'] = entropy(list(counter.values()))
                else:
                    record['mode_prediction'] = None
                    record['prediction_entropy'] = None
            
            summary_records.append(record)
        
        summary_df = pd.DataFrame(summary_records)
        output_file = f'{RESULTS_DIR}/{task}_{model_name}_ft_{ft_label}_summary.csv'
        summary_df.to_csv(output_file, index=False)
        print(f"Saved: {output_file}")
        return summary_df
    
    return None  # MatKG requires separate processing

In [ ]:
# Example: Aggregate results
# summary = aggregate_gpt_results('Bandgap', 'gpt-4o', 'base', n_iterations=10)
# summary = aggregate_gpt_results('Bandgap', 'gpt-4o', 'Bandgap', n_iterations=10)

## 10. Cost Estimation

Estimate the cost of finetuning and inference.

In [ ]:
def estimate_finetuning_cost(task, base_model='gpt-4o', n_epochs=3):
    """
    Estimate finetuning cost based on training data size.
    
    Pricing (as of 2024, check OpenAI for current rates):
    - GPT-4o: $25/1M training tokens
    - GPT-4o-mini: $3/1M training tokens
    - GPT-3.5-turbo: $8/1M training tokens
    """
    pricing = {
        'gpt-4o': 25.0,
        'gpt-4o-mini': 3.0,
        'gpt-4.1': 25.0,
        'gpt-4.1-mini': 3.0,
        'gpt-3.5-turbo': 8.0
    }
    
    # Rough estimate: ~500 tokens per training example
    train_df = pd.read_csv(TRAIN_DATA[task])
    n_examples = len(train_df)
    tokens_per_example = 500
    total_tokens = n_examples * tokens_per_example * n_epochs
    
    cost_per_m = pricing.get(base_model, 10.0)
    estimated_cost = (total_tokens / 1_000_000) * cost_per_m
    
    print(f"Task: {task}")
    print(f"Training examples: {n_examples}")
    print(f"Estimated tokens: {total_tokens:,}")
    print(f"Estimated cost for {base_model}: ${estimated_cost:.2f}")
    
    return estimated_cost

# Estimate costs for all tasks
# for task in ['Bandgap', 'Dielectric', 'CrystalStructure', 'MatKG']:
#     estimate_finetuning_cost(task, 'gpt-4o-mini')
#     print()

## Notes

### Workflow Summary

1. **Prepare data**: Run `prepare_finetuning_data()` for each task
2. **Upload files**: Run `upload_training_file()` for each JSONL file
3. **Start finetuning**: Run `create_finetuning_job()` with your file IDs
4. **Wait for completion**: Use `wait_for_job()` or check status periodically
5. **Save model IDs**: Store finetuned model IDs in `finetuned_model_ids.json`
6. **Run inference**: Use `run_gpt_experiment()` for base and finetuned models
7. **Aggregate results**: Use `aggregate_gpt_results()` to create summaries

### Important Considerations

- **Cost**: GPT finetuning and inference costs money. Estimate costs before running.
- **Rate limits**: Add delays between API calls to avoid hitting rate limits.
- **Model availability**: Finetuned models are private to your OpenAI account.
- **Data format**: Ensure training data matches the expected JSONL format.

### Differences from Open-Source Models

- GPT uses system/user message format (not prompt templates)
- Finetuning is done through OpenAI API (not Predibase/HuggingFace)
- No access to model weights or embeddings
- Cannot share finetuned models publicly